# 05.24 - Hyperparameter Tuning with Optuna

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Manual grid search is slow and wasteful. Optuna uses Bayesian optimization to find better hyperparameters faster.

## 2. Why Does This Matter?

Good hyperparameters can improve model performance by 5-20%. Optuna does this in a fraction of the time.

## 3. Prerequisites

- 05.17: Model selection

## 4. Learning Objectives

- Install and use Optuna for hyperparameter tuning
- Define search spaces
- Use pruning to stop bad trials early
- Compare with GridSearchCV

## 5. Mental Model

Optuna = Bayesian optimization + pruning + visualization.
It learns from previous trials to suggest better parameters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries loaded.")

## 6. Install Optuna

In [ ]:
try:
    import optuna
    print("Optuna version: " + optuna.__version__)
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna
    print("Optuna installed: " + optuna.__version__)

## 7. Define the Objective Function

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    
    rf = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
        max_features=max_features, random_state=42
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(rf, X, y, cv=cv, scoring="accuracy")
    return scores.mean()

## 8. Run the Study

In [ ]:
study = optuna.create_study(direction="maximize", study_name="wine_rf")
study.optimize(objective, n_trials=30, show_progress_bar=False)

print("Best trial:")
print("  Accuracy: " + str(round(study.best_trial.value, 4)))
print("  Params: " + str(study.best_trial.params))

## 9. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trials = study.trials
vals = [t.value for t in trials]
best_so_far = [max(vals[:i+1]) for i in range(len(vals))]
axes[0].plot(range(len(vals)), vals, "b.", alpha=0.5, label="Trial")
axes[0].plot(range(len(best_so_far)), best_so_far, "r-", label="Best so far")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Optimization History")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Param importance
try:
    importance = optuna.importance.get_param_importances(study)
    names = list(importance.keys())[:6]
    vals_imp = [importance[n] for n in names]
    axes[1].barh(names, vals_imp)
    axes[1].set_xlabel("Importance")
    axes[1].set_title("Hyperparameter Importance")
except Exception:
    axes[1].text(0.5, 0.5, "Importance not available", ha="center", va="center")

plt.tight_layout()
plt.savefig("optuna_results.png", dpi=100, bbox_inches="tight")
plt.show()

## 10. Compare with Best Model

In [ ]:
best = study.best_trial
best_rf = RandomForestClassifier(**best.params, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(best_rf, X, y, cv=cv, scoring="accuracy")
print("Tuned RF accuracy: " + str(round(scores.mean(), 4)) + " (+/- " + str(round(scores.std(), 4)) + ")")

default_rf = RandomForestClassifier(random_state=42)
scores_default = cross_val_score(default_rf, X, y, cv=cv, scoring="accuracy")
print("Default RF accuracy: " + str(round(scores_default.mean(), 4)) + " (+/- " + str(round(scores_default.std(), 4)) + ")")

## 11. Common Mistakes

1. Too few trials (need 50+ for good results)
2. Too wide search space
3. Not using pruning
4. Overfitting to validation set

## 12. Coding Exercises

### Exercise 1: Tune XGBoost
Use Optuna to tune XGBoost.

### Exercise 2: Multi-objective
Optimize for both accuracy and model size.

In [ ]:
# EXERCISE 1: Tune XGBoost
try:
    from xgboost import XGBClassifier
    print("Exercise: Optimize XGBoost with Optuna.")
except ImportError:
    print("XGBoost not installed.")

In [ ]:
# EXERCISE 2: Multi-objective optimization
print("Exercise: Optimize accuracy AND inference time.")

## 13. Closed-Book Recall

1. What makes Optuna faster than grid search?
2. What is pruning?
3. How do you define a search space?

## 14. Teach-Back Questions

Explain Bayesian optimization vs grid search. How pruning saves time.

## 15. Summary

Optuna uses Bayesian optimization to find hyperparameters efficiently. Pruning stops bad trials early. Visualizations help understand the search space.

## 16. Further Experiment

1. Try different samplers (TPE, CMA-ES).
2. Use Optuna with PyTorch.
3. Create a reusable tuning template.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [numpy, pandas, matplotlib, scikit-learn, optuna]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```